# IPF bulk RNA-seq workflow notebook

This notebook mirrors the assignment workflow for **idiopathic pulmonary fibrosis (IPF)** and is designed to be practical.  
It covers:

1. raw-read quality control with **FastQC** and **MultiQC**
2. adapter/quality trimming with **fastp**
3. decoy-aware transcript quantification with **Salmon selective alignment**
4. gene-level import with **tximport**
5. differential expression with **DESeq2**
6. pathway analysis with **fgsea** and **gprofiler2**

The shell workflow is also provided separately as `run_ipf_rnaseq_workflow.sh`.  
This notebook is useful for the same analysis in a more readable, stepwise format with inline comments.


In [ ]:
from pathlib import Path
import gzip
import os
import shutil
import subprocess
import textwrap
import pandas as pd

# -----------------------------#
# User-editable project paths  #
# -----------------------------#
# Change these paths to match your own project structure.
PROJECT_DIR = Path(".").resolve()
READS_DIR = PROJECT_DIR / "fastq"
REF_DIR = PROJECT_DIR / "ref"
METADATA_CSV = PROJECT_DIR / "metadata" / "ipf_metadata.csv"
OUT_DIR = PROJECT_DIR / "analysis_output"

TRIM_DIR = OUT_DIR / "trimmed_fastq"
QC_RAW_DIR = OUT_DIR / "qc_raw"
QC_TRIMMED_DIR = OUT_DIR / "qc_trimmed"
MULTIQC_RAW_DIR = OUT_DIR / "multiqc_raw"
MULTIQC_TRIM_DIR = OUT_DIR / "multiqc_trimmed"
QUANT_DIR = OUT_DIR / "salmon_quant"
R_RESULTS_DIR = OUT_DIR / "r_results"
LOG_DIR = OUT_DIR / "logs"

GENOME_FASTA = REF_DIR / "GRCh38.primary_assembly.genome.fa"
TRANSCRIPTS_FASTA = REF_DIR / "gencode.transcripts.fa.gz"
GTF_FILE = REF_DIR / "gencode.annotation.gtf.gz"
TX2GENE_TSV = REF_DIR / "tx2gene.tsv"

DECOYS_TXT = REF_DIR / "decoys.txt"
GENTROME_FASTA = REF_DIR / "gentrome.fa"
SALMON_INDEX = REF_DIR / "salmon_grch38_gencode_decoy"

THREADS = 8

for path in [TRIM_DIR, QC_RAW_DIR, QC_TRIMMED_DIR, MULTIQC_RAW_DIR, MULTIQC_TRIM_DIR, QUANT_DIR, R_RESULTS_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Output directory:  {OUT_DIR}")


In [ ]:
def run_command(cmd, log_file=None):
    """Run a shell command and stream output. Raise an error if it fails."""
    print(" ".join(map(str, cmd)))
    if log_file is None:
        subprocess.run(cmd, check=True)
    else:
        with open(log_file, "w", encoding="utf-8") as handle:
            subprocess.run(cmd, check=True, stdout=handle, stderr=subprocess.STDOUT)

def assert_required_files():
    required = [GENOME_FASTA, TRANSCRIPTS_FASTA, GTF_FILE, TX2GENE_TSV, METADATA_CSV]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required input(s):\n" + "\n".join(missing))

    r1_files = sorted(READS_DIR.glob("*_R1.fastq.gz"))
    if not r1_files:
        raise FileNotFoundError(f"No *_R1.fastq.gz files found in {READS_DIR}")

    for r1 in r1_files:
        sample = r1.name.replace("_R1.fastq.gz", "")
        r2 = READS_DIR / f"{sample}_R2.fastq.gz"
        if not r2.exists():
            raise FileNotFoundError(f"Missing mate-pair FASTQ for sample: {sample}")

    print("All required reference, metadata, and FASTQ inputs were found.")

assert_required_files()


In [ ]:
# ------------------------------#
# 1. Validate the sample sheet   #
# ------------------------------#
# This keeps the downstream design formula transparent.
metadata = pd.read_csv(METADATA_CSV)
required_columns = {"sample_id", "condition", "batch", "sex"}
missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise ValueError(f"Metadata is missing required columns: {sorted(missing_columns)}")

display(metadata.head())
print(metadata["condition"].value_counts(dropna=False))


In [ ]:
# -----------------------------------------#
# 2. Raw-read QC with FastQC and MultiQC    #
# -----------------------------------------#
for r1 in sorted(READS_DIR.glob("*_R1.fastq.gz")):
    sample = r1.name.replace("_R1.fastq.gz", "")
    r2 = READS_DIR / f"{sample}_R2.fastq.gz"

    run_command([
        "fastqc",
        "--threads", str(THREADS),
        "--outdir", str(QC_RAW_DIR),
        str(r1), str(r2)
    ])

run_command([
    "multiqc",
    str(QC_RAW_DIR),
    "--outdir", str(MULTIQC_RAW_DIR),
    "--filename", "multiqc_raw_reads.html"
])


In [ ]:
# ----------------------------------------------#
# 3. Trim reads with fastp                       #
# ----------------------------------------------#
# Settings chosen to match the assignment narrative:
# - paired-end adapter detection
# - Q20 threshold
# - minimum read length of 35 nt
# - polyG/polyX trimming for NovaSeq-style low-complexity tails
for r1 in sorted(READS_DIR.glob("*_R1.fastq.gz")):
    sample = r1.name.replace("_R1.fastq.gz", "")
    r2 = READS_DIR / f"{sample}_R2.fastq.gz"

    out_r1 = TRIM_DIR / f"{sample}_R1.trimmed.fastq.gz"
    out_r2 = TRIM_DIR / f"{sample}_R2.trimmed.fastq.gz"
    html_report = TRIM_DIR / f"{sample}.fastp.html"
    json_report = TRIM_DIR / f"{sample}.fastp.json"
    log_file = LOG_DIR / f"{sample}.fastp.log"

    run_command([
        "fastp",
        "--in1", str(r1),
        "--in2", str(r2),
        "--out1", str(out_r1),
        "--out2", str(out_r2),
        "--detect_adapter_for_pe",
        "--qualified_quality_phred", "20",
        "--length_required", "35",
        "--trim_poly_g",
        "--trim_poly_x",
        "--thread", str(THREADS),
        "--html", str(html_report),
        "--json", str(json_report)
    ], log_file=log_file)

# Re-run FastQC after trimming and aggregate the reports.
for r1 in sorted(TRIM_DIR.glob("*_R1.trimmed.fastq.gz")):
    sample = r1.name.replace("_R1.trimmed.fastq.gz", "")
    r2 = TRIM_DIR / f"{sample}_R2.trimmed.fastq.gz"

    run_command([
        "fastqc",
        "--threads", str(THREADS),
        "--outdir", str(QC_TRIMMED_DIR),
        str(r1), str(r2)
    ])

run_command([
    "multiqc",
    str(QC_TRIMMED_DIR),
    str(TRIM_DIR),
    "--outdir", str(MULTIQC_TRIM_DIR),
    "--filename", "multiqc_trimmed_reads.html"
])


In [ ]:
# ---------------------------------------------------------#
# 4. Build a decoy-aware Salmon index                       #
# ---------------------------------------------------------#
# Salmon selective alignment works best with a decoy-aware
# gentrome (transcriptome + genomic decoy sequences).

if not (SALMON_INDEX / "versionInfo.json").exists():
    # Write decoy sequence names from the reference genome FASTA.
    with open(GENOME_FASTA, "rt", encoding="utf-8", errors="ignore") as genome_in, open(DECOYS_TXT, "w", encoding="utf-8") as decoys_out:
        for line in genome_in:
            if line.startswith(">"):
                decoys_out.write(line[1:].split()[0] + "\n")

    # Concatenate transcriptome FASTA and genome FASTA into a gentrome.
    with open(GENTROME_FASTA, "wb") as gentrome_out:
        with gzip.open(TRANSCRIPTS_FASTA, "rb") as tx_in:
            shutil.copyfileobj(tx_in, gentrome_out)
        with open(GENOME_FASTA, "rb") as genome_in:
            shutil.copyfileobj(genome_in, gentrome_out)

    run_command([
        "salmon", "index",
        "--transcripts", str(GENTROME_FASTA),
        "--decoys", str(DECOYS_TXT),
        "--index", str(SALMON_INDEX),
        "--kmerLen", "31",
        "--threads", str(THREADS)
    ])
else:
    print("Existing Salmon index found. Reusing it.")


In [ ]:
# ---------------------------------------------------------#
# 5. Quantify each sample with Salmon                       #
# ---------------------------------------------------------#
# Key nondefault settings:
# - explicit library type: ISR
# - validateMappings
# - GC, sequence, and positional bias correction
for r1 in sorted(TRIM_DIR.glob("*_R1.trimmed.fastq.gz")):
    sample = r1.name.replace("_R1.trimmed.fastq.gz", "")
    r2 = TRIM_DIR / f"{sample}_R2.trimmed.fastq.gz"
    sample_out = QUANT_DIR / sample
    sample_log = LOG_DIR / f"{sample}.salmon.log"

    run_command([
        "salmon", "quant",
        "--index", str(SALMON_INDEX),
        "--libType", "ISR",
        "--mates1", str(r1),
        "--mates2", str(r2),
        "--output", str(sample_out),
        "--threads", str(THREADS),
        "--validateMappings",
        "--seqBias",
        "--gcBias",
        "--posBias"
    ], log_file=sample_log)

run_command([
    "multiqc",
    str(QUANT_DIR),
    str(LOG_DIR),
    "--outdir", str(OUT_DIR),
    "--filename", "multiqc_quantification_summary.html"
])


In [ ]:
# ---------------------------------------------------------#
# 6. Save the downstream R analysis script                  #
# ---------------------------------------------------------#
# The DESeq2 / fgsea / gprofiler2 steps are written out as
# a standalone R script so the workflow remains portable.

r_script = r'''suppressPackageStartupMessages({
  library(tximport)
  library(readr)
  library(dplyr)
  library(DESeq2)
  library(apeglm)
  library(fgsea)
  library(msigdbr)
  library(gprofiler2)
  library(pheatmap)
  library(ggplot2)
})

metadata_csv <- Sys.getenv("METADATA_CSV")
tx2gene_tsv <- Sys.getenv("TX2GENE_TSV")
quant_dir <- Sys.getenv("QUANT_DIR")
results_dir <- Sys.getenv("R_RESULTS_DIR")

dir.create(results_dir, recursive = TRUE, showWarnings = FALSE)

#-----------------------------#
# 1. Read metadata and files  #
#-----------------------------#
meta <- read_csv(metadata_csv, show_col_types = FALSE) %>%
  mutate(
    condition = factor(condition, levels = c("Control", "IPF")),
    batch = factor(batch),
    sex = factor(sex)
  )

required_cols <- c("sample_id", "condition", "batch", "sex")
missing_cols <- setdiff(required_cols, names(meta))
if (length(missing_cols) > 0) {
  stop("Metadata is missing required columns: ", paste(missing_cols, collapse = ", "))
}

files <- file.path(quant_dir, meta$sample_id, "quant.sf")
names(files) <- meta$sample_id

if (any(!file.exists(files))) {
  stop("Missing Salmon quant.sf files for: ",
       paste(names(files)[!file.exists(files)], collapse = ", "))
}

tx2gene <- read_tsv(tx2gene_tsv, show_col_types = FALSE, col_names = c("TXNAME", "GENEID"))

#-----------------------------------------------#
# 2. Import transcript estimates at gene level  #
#-----------------------------------------------#
txi <- tximport(
  files = files,
  type = "salmon",
  tx2gene = tx2gene,
  countsFromAbundance = "lengthScaledTPM",
  ignoreTxVersion = TRUE
)

#---------------------------------------#
# 3. Differential expression in DESeq2  #
#---------------------------------------#
dds <- DESeqDataSetFromTximport(
  txi = txi,
  colData = as.data.frame(meta),
  design = ~ batch + sex + condition
)

# Prefilter low-information genes:
# keep genes with >=10 counts in at least 25% of samples.
keep <- rowSums(counts(dds) >= 10) >= ceiling(0.25 * ncol(dds))
dds <- dds[keep, ]

dds <- DESeq(dds)

res <- results(dds, contrast = c("condition", "IPF", "Control"))
res_shrunk <- lfcShrink(dds, coef = "condition_IPF_vs_Control", type = "apeglm")

res_tbl <- as.data.frame(res) %>%
  tibble::rownames_to_column("gene_id")
res_shrunk_tbl <- as.data.frame(res_shrunk) %>%
  tibble::rownames_to_column("gene_id")

write_csv(res_tbl, file.path(results_dir, "deseq2_results_raw.csv"))
write_csv(res_shrunk_tbl, file.path(results_dir, "deseq2_results_shrunken.csv"))

#--------------------------------#
# 4. Exploratory sample profiles #
#--------------------------------#
vsd <- vst(dds, blind = FALSE)
vst_mat <- assay(vsd)

# PCA
pca_df <- plotPCA(vsd, intgroup = c("condition", "batch"), returnData = TRUE)
percent_var <- round(100 * attr(pca_df, "percentVar"))

p <- ggplot(pca_df, aes(PC1, PC2, color = condition, shape = batch)) +
  geom_point(size = 3) +
  xlab(paste0("PC1: ", percent_var[1], "% variance")) +
  ylab(paste0("PC2: ", percent_var[2], "% variance")) +
  ggtitle("IPF bulk RNA-seq PCA") +
  theme_bw(base_size = 11)

ggsave(file.path(results_dir, "pca_condition_batch.png"), plot = p, width = 7, height = 5, dpi = 300)

# Sample distance heatmap
sample_dists <- dist(t(vst_mat))
sample_dist_mat <- as.matrix(sample_dists)
png(file.path(results_dir, "sample_distance_heatmap.png"), width = 1800, height = 1500, res = 220)
pheatmap(sample_dist_mat, clustering_distance_rows = sample_dists, clustering_distance_cols = sample_dists)
dev.off()

# MA plot
png(file.path(results_dir, "ma_plot.png"), width = 1800, height = 1500, res = 220)
plotMA(res_shrunk, ylim = c(-5, 5), main = "IPF vs Control (shrunken log2FC)")
dev.off()

#-------------------------------------#
# 5. Ranking for pathway-level GSEA   #
#-------------------------------------#
wald_rank <- res_tbl$stat
names(wald_rank) <- res_tbl$gene_id
wald_rank <- sort(na.omit(wald_rank), decreasing = TRUE)

hallmark_df <- msigdbr(species = "Homo sapiens", category = "H") %>%
  select(gs_name, gene_symbol)

reactome_df <- msigdbr(species = "Homo sapiens", category = "C2", subcategory = "CP:REACTOME") %>%
  select(gs_name, gene_symbol)

hallmark_pathways <- split(hallmark_df$gene_symbol, hallmark_df$gs_name)
reactome_pathways <- split(reactome_df$gene_symbol, reactome_df$gs_name)

fgsea_h <- fgseaMultilevel(
  pathways = hallmark_pathways,
  stats = wald_rank,
  minSize = 15,
  maxSize = 500
) %>% arrange(padj, desc(abs(NES)))

fgsea_r <- fgseaMultilevel(
  pathways = reactome_pathways,
  stats = wald_rank,
  minSize = 15,
  maxSize = 500
) %>% arrange(padj, desc(abs(NES)))

write_csv(as.data.frame(fgsea_h), file.path(results_dir, "fgsea_hallmark.csv"))
write_csv(as.data.frame(fgsea_r), file.path(results_dir, "fgsea_reactome.csv"))

#-------------------------------------------------------#
# 6. Over-representation analysis with gprofiler2       #
#-------------------------------------------------------#
sig_genes <- res_shrunk_tbl %>%
  filter(!is.na(padj), padj < 0.05, abs(log2FoldChange) >= 1) %>%
  pull(gene_id)

tested_genes <- res_tbl %>%
  filter(!is.na(baseMean)) %>%
  pull(gene_id)

if (length(sig_genes) > 0) {
  gost_res <- gost(
    query = sig_genes,
    organism = "hsapiens",
    sources = c("GO:BP", "REAC", "KEGG"),
    correction_method = "fdr",
    custom_bg = tested_genes
  )

  if (!is.null(gost_res$result)) {
    write_csv(as.data.frame(gost_res$result), file.path(results_dir, "gprofiler_sig_genes.csv"))
  }
}

#---------------------------------------------#
# 7. Save normalized counts for downstream use #
#---------------------------------------------#
norm_counts <- counts(dds, normalized = TRUE) %>%
  as.data.frame() %>%
  tibble::rownames_to_column("gene_id")

write_csv(norm_counts, file.path(results_dir, "normalized_counts.csv"))
saveRDS(dds, file.path(results_dir, "dds_object.rds"))
saveRDS(vsd, file.path(results_dir, "vsd_object.rds"))

message("Downstream DE and enrichment analysis completed.")'''
r_script_path = PROJECT_DIR / "ipf_deseq2_fgsea_gprofiler2.R"
r_script_path.write_text(r_script, encoding="utf-8")
print(f"Saved R script to: {r_script_path}")


In [ ]:
# ---------------------------------------------------------#
# 7. Run the downstream R analysis                          #
# ---------------------------------------------------------#
env = os.environ.copy()
env["METADATA_CSV"] = str(METADATA_CSV)
env["TX2GENE_TSV"] = str(TX2GENE_TSV)
env["QUANT_DIR"] = str(QUANT_DIR)
env["R_RESULTS_DIR"] = str(R_RESULTS_DIR)

subprocess.run(
    ["Rscript", str(PROJECT_DIR / "ipf_deseq2_fgsea_gprofiler2.R")],
    check=True,
    env=env
)


## What this notebook gives you

- a reproducible preprocessing path from FASTQ to `quant.sf`
- a DESeq2 design consistent with the written narrative: `~ batch + sex + condition`
- pathway analysis settings consistent with the assignment write-up
- code that is readable enough for grading and still usable outside the classroom, which is rare enough to qualify as a minor miracle
